# Add new parameters and constraints in ES

In [1]:
import pandas as pd
from shared.utils import load_snapshot
from energyscope.models import Model

YEAR: year of optimization
TEC: technology
MAT: material

Parameters:
- `material_intensity[YEAR, TEC, MAT]` [kt / GW]
- `limit_material_year[YEAR, MAT]` [kt]
- `limit_material[MAT]` [kt]
- `recycling_rate[YEAR, (TEC), MAT]` [-]

Variables:
- `Material_content_year[YEAR, TEC, MAT]` [kt]
- `Material_content[TEC, MAT]` [kt]
- `Recycled_material[YEAR, TEC, MAT]` [kt]

Constraints:
- `Material_content_year[YEAR, TEC, MAT] = material_intensity[YEAR, TEC, MAT] * F_new[YEAR, TEC, MAT]`
- ...

In [2]:
data = [
    ['material_intensity', 'YEAR_2020', 'WIND_ONSHORE', 'Li', 0.0013, 'kt/GW', 'a comment'],
]
df = pd.DataFrame(data, columns=['Parameter', 'index0', 'index1', 'index2', 'Value', 'Unit', 'Comment'])

In [3]:
df = pd.read_excel("excel_files/technologies_mi_all_years.xlsx")

In [4]:
def create_dat_file_from_excel(df, file_name):
    out_path = f'ampl_files/{file_name}.dat'
    # use utf-8-sig so Windows Notepad shows accents correctly; use 'utf-8' if BOM is not desired
    with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
        f.write("data;\n\n")
        f.write("set MATERIALS := Al B Cd Cr Co Concrete Cu Dy Ga Glass Ge Hf In Fe Pb Polymers Li Mg Mn Mo Nd Ni Nb Pr Se Si Ag Ta Te Tb Sn W V Y Zn Zr ;\n \n")
        for _, row in df.iterrows():
            value = row['Value']
            if pd.isna(value):
                continue  # skip missing values, params already default to 0
            param_name = row['Parameter']
            index0 = row['index0']
            index1 = row['index1']
            index2 = row['index2']
            unit = '-' if pd.isna(row.get('Unit')) else str(row.get('Unit'))
            comment = '' if pd.isna(row.get('Comment')) else str(row.get('Comment'))
            if pd.isna(index1) and pd.isna(index2):
                f.write(f"let {param_name}['{index0}'] := {value} ; # [{unit}] {comment}\n")
            elif pd.isna(index2):
                f.write(f"let {param_name}['{index0}','{index1}'] := {value} ; # [{unit}] {comment}\n")
            else:
                f.write(f"let {param_name}['{index0}','{index1}','{index2}'] := {value} ; # [{unit}] {comment}\n")


In [5]:
create_dat_file_from_excel(df, 'Material_intensity')

In [6]:
# Add your files to the main model
main_model = load_snapshot(2050) + Model([
    ('mod', 'ampl_files/test.mod'),
    ('dat', 'ampl_files/test.dat'),
])

## Voir la demande en matériau après un run

Charge `_Materials_Results.pkl` produit par `src/run_pathway_materials.py` (indexé par `Years, Technologies, Materials`).

In [19]:
import pickle

with open('out/Materials_test/_Materials_Results.pkl', 'rb') as f:
    materials_results = pickle.load(f)

mcy = materials_results['Material_content_year']['Material_content_year']
recycled = materials_results['Recycled_material']['Recycled_material']

# Demande annuelle par materiau, toutes technologies confondues [kt/an]
demand_by_year_material = mcy.groupby(['Years', 'Materials']).sum()
demand_by_year_material = demand_by_year_material[demand_by_year_material != 0].sort_values(ascending=False)
#demand_by_year_material.head(20)

In [20]:
# Detail par technologie pour un materiau donne, par exemple 'Li'
material = 'Nd'
mcy.xs(material, level='Materials')[lambda s: s != 0].unstack('Technologies')

Technologies,WIND_ONSHORE
Years,
YEAR_2025,8.899907
YEAR_2030,12.344266
YEAR_2035,102.755827


In [12]:
import pickle
with open('out/Materials_test/_Results.pkl', 'rb') as f:
    results = pickle.load(f)

f_new = results['F_new']
f_new.xs('WIND_ONSHORE', level='Technologies')   # ou 'PV_ROOF', 'NEW_HYDRO_DAM'

,F_new
Phases,
2015_2020,3.9460
2020_2025,0.8352
2025_2030,0.0000
2030_2035,7.5864
2035_2040,0.7892
2040_2045,0.7892
2045_2050,0.8352


In [18]:
import pickle
with open('out/Materials_test/_Results.pkl', 'rb') as f:
    results = pickle.load(f)

f_new = results['F_new']
f_new.xs('WIND_ONSHORE', level='Technologies')   # ou 'PV_ROOF', 'NEW_HYDRO_DAM'

,F_new
Phases,
2015_2020,3.946001
2020_2025,0.835199
2025_2030,1.122789
2030_2035,8.877211
2035_2040,0.000000
2040_2045,0.000000
2045_2050,0.000000
